# 02 — Phase 2: Chronological Split + CV Folds (merged)

**Purpose.** Runs the two merged Phase 2 SQL files and checks their results:

- `sql/003_phase2_split.sql` builds `monthly_split` from `county_month_joined` (Train through 2021-09-30, Embargo 2021-Q4, Test 2022-01-01 to 2026-03-31, everything after as `unassigned`) and stores Jolenn's 11 expanding-window CV folds in `cv_folds`, with consistency checks.
- `sql/003b_cv_fold_membership.sql` builds the `cv_fold_rows` view, which assigns each Train county-quarter to each fold as `train` or `val`.

**Prerequisites.** Phase 1 is complete (`county_month_joined` exists). For section 6, `sql/005_phase4_features.sql` must already have been run, since `003b` reads `quarterly_modeling_table`. This notebook does not run `005`.

## 1. Confirm the kernel

In [1]:
import sys
import pandas as pd
print(sys.executable)

d:\Capstone\capstone-housing-risk-team\.venv\Scripts\python.exe


## 2. Connect to Postgres

Same convention as the earlier notebooks: credentials come from the environment via `housing_risk.config`.

In [2]:
import os
from pathlib import Path

from sqlalchemy import create_engine, text

try:
    from housing_risk.config import get_engine
    engine = get_engine()
except ImportError:
    # No housing_risk package in this repo: build the engine from .env
    # (PGUSER / PGPASSWORD / PGHOST / PGPORT / PGDATABASE), same variables as before.
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
    engine = create_engine(
        "postgresql+psycopg2://{u}:{p}@{h}:{port}/{d}".format(
            u=os.environ["PGUSER"], p=os.environ["PGPASSWORD"],
            h=os.environ.get("PGHOST", "localhost"),
            port=os.environ.get("PGPORT", "5432"),
            d=os.environ.get("PGDATABASE", "housing_risk_team"),
        )
    )

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version();")).fetchone())
    n = conn.execute(text("SELECT COUNT(*) FROM county_month_joined")).scalar()
print(f"county_month_joined: {n:,} rows")

('housing_risk_team', 'PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit')
county_month_joined: 709,996 rows


## 3. Run `003_phase2_split.sql`

The script contains a `DO $$ ... $$` block and `%` characters inside `RAISE NOTICE`, so it is sent through the raw DBAPI cursor with no parameters instead of `exec_driver_sql`. The `DO` block raises an error, and this cell stops, if any fold check fails (validation reaching past Train, train overlapping validation, folds not expanding, or wrong quarter counts).

In [3]:
def find_sql(name):
    # Works whether the notebook runs from notebooks/ (VS Code default) or the repo root.
    for base in (Path("sql"), Path("..") / "sql"):
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f"{name} not found in ./sql or ../sql")


def run_sql_file(name):
    path = find_sql(name)
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.execute(open(path).read())
        raw.commit()
        for notice in getattr(raw, "notices", []):
            print(notice.strip())
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()

run_sql_file("003_phase2_split.sql")

NOTICE:  table "monthly_split" does not exist, skipping
NOTICE:  table "cv_folds" does not exist, skipping
NOTICE:  Phase 2 checks passed: 11 folds, all inside Train (last train month 2021-09-30)


## 4. Row counts per split

Compared against the monthly figures in the original `003` comment. A mismatch is not a failure of the split logic; it points at a data-coverage difference between the two ZHVI downloads. The unassigned bucket should be 5 months × 3,071 counties = 15,355.

In [4]:
counts = pd.read_sql(
    """
    SELECT split, COUNT(*) AS n_rows, COUNT(DISTINCT county_fips) AS n_counties,
           MIN(month) AS first_month, MAX(month) AS last_month
    FROM monthly_split GROUP BY split ORDER BY split
    """, engine)

documented = {"train": 528_174, "embargo": 9_063, "test": 156_361, "unassigned": 15_355}
counts["documented_n_rows"] = counts["split"].map(documented)
counts["diff"] = counts["n_rows"] - counts["documented_n_rows"]
print(f"Total: {counts['n_rows'].sum():,} rows (documented total {sum(documented.values()):,})")
counts

Total: 709,996 rows (documented total 708,953)


,split,n_rows,n_counties,first_month,last_month,documented_n_rows,diff
0,embargo,9063,3021,2021-10-31,2021-12-31,9063,0
1,test,156364,3071,2022-01-31,2026-03-31,156361,3
2,train,529214,3021,2000-01-31,2021-09-30,528174,1040
3,unassigned,15355,3071,2026-04-30,2026-08-31,15355,0


## 5. Inspect the CV folds

In [5]:
folds = pd.read_sql("SELECT * FROM cv_folds ORDER BY fold_id", engine)
folds

,fold_id,train_start,train_end,val_start,val_end,n_train_quarters,n_val_quarters
0,0,2000-03-31,2009-12-31,2010-03-31,2010-12-31,40,4
1,1,2000-03-31,2010-12-31,2011-03-31,2011-12-31,44,4
2,2,2000-03-31,2011-12-31,2012-03-31,2012-12-31,48,4
3,3,2000-03-31,2012-12-31,2013-03-31,2013-12-31,52,4
4,4,2000-03-31,2013-12-31,2014-03-31,2014-12-31,56,4
5,5,2000-03-31,2014-12-31,2015-03-31,2015-12-31,60,4
6,6,2000-03-31,2015-12-31,2016-03-31,2016-12-31,64,4
7,7,2000-03-31,2016-12-31,2017-03-31,2017-12-31,68,4
8,8,2000-03-31,2017-12-31,2018-03-31,2018-12-31,72,4
9,9,2000-03-31,2018-12-31,2019-03-31,2019-12-31,76,4


## 6. Run `003b_cv_fold_membership.sql`

Fold dates are quarter-end; `quarterly_modeling_table` stores quarter start, and the view converts before comparing. Only Train-split rows are eligible for a fold.

In [ ]:
with engine.connect() as conn:
    exists = conn.execute(text("SELECT to_regclass('quarterly_modeling_table')")).scalar()
assert exists, "STOP: quarterly_modeling_table not found -- run sql/005_phase4_features.sql first."

run_sql_file("003b_cv_fold_membership.sql")

In [ ]:
fold_summary = pd.read_sql(
    """
    SELECT fold_id, fold_role, COUNT(*) AS n_rows,
           SUM(decline_label) AS n_positives,
           ROUND(100.0 * AVG(decline_label), 2) AS positive_rate_pct
    FROM cv_fold_rows GROUP BY fold_id, fold_role ORDER BY fold_id, fold_role
    """, engine)

MIN_FOLD_POSITIVES = 20  # from Jolenn's baseline_lock.json
val = fold_summary[fold_summary["fold_role"] == "val"]
low = val[val["n_positives"] < MIN_FOLD_POSITIVES]
print(f"{len(val)} folds with validation rows; {len(low)} below {MIN_FOLD_POSITIVES} positives")
fold_summary

In [ ]:
# Leakage guard: no Embargo/Test rows in any fold.
leak = pd.read_sql(
    """
    SELECT COUNT(*) FROM cv_fold_rows r
    JOIN quarterly_modeling_table m USING (county_fips, quarter)
    WHERE m.split <> 'train'
    """, engine).iloc[0, 0]
assert leak == 0, f"STOP: {leak} fold rows come from Embargo/Test"
print("No Embargo/Test rows appear in any CV fold.")